# SETU-RAG — Speech-to-Speech Demo
**Code-switching-aware multilingual RAG** for 22 Indian languages + English.  
Speak (or upload audio) in any mixture of Hindi/Tamil/Bengali/… + English → get a spoken answer.

> **Before running:** Runtime → Change runtime type → **T4 GPU** → Save

## 1 · Setup

In [ ]:
import os

# Get the code
if not os.path.isdir('setu-rag'):
    !git clone https://github.com/Gaurs86/setu-rag.git
os.chdir('setu-rag')

# Core deps
!pip -q install -r colab_requirements.txt

# Speech deps — parler-tts pulls protobuf 4.x as a transitive dep, so install it first
!pip -q install soundfile librosa
!pip -q install git+https://github.com/huggingface/parler-tts.git

# Pin protobuf LAST (after parler-tts may have downgraded it).
# transformers >= 4.47 needs google.protobuf.runtime_version (added in protobuf 5.26).
# The 5.29.x line satisfies transformers AND Colab's tensorflow/ydf/grain/grpcio,
# so we cap below 6.0 to avoid the protobuf-6 conflicts those packages raise.
# The pip "dependency conflict" lines that may still print (e.g. descript-audiotools
# wanting protobuf<5) are non-fatal warnings — they don't affect the SETU-RAG runtime.
# No kernel restart: a fresh Colab kernel hasn't imported transformers yet, so the new
# packages take effect on the first import. Restarting lets Colab revert its cached packages.
!pip -q install "protobuf>=5.29.0,<6.0.0"

print('✓ ready — run the cells below (do not re-run this cell)')

In [ ]:
# Optional — HF token for gated models (Sarvam-1, Aya-Expanse, IndicConformer, Parler-TTS)
# Add HF_TOKEN via the 🔑 Secrets panel (left sidebar) and accept each model's license on HF.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login; login(os.environ['HF_TOKEN'])
    print('HF login OK')
except Exception:
    print('No HF token — running with un-gated models (fine for the default demo)')

## 2 · Build the RAG pipeline
Downloads ~3–4 GB of weights on the first run (BGE-M3 embedder, BGE-reranker, Qwen2.5-3B-Instruct).  
Later runs are fast — weights stay cached in `~/.cache/huggingface`.

In [ ]:
import sys; sys.path.insert(0, os.getcwd())
from setu_rag.app import build_pipeline

rag = build_pipeline()   # device='cuda', 4-bit generator, real BGE-M3 embedder
print('Pipeline ready')

## 3 · Speech-to-speech (mic in → spoken answer)
**Record or upload** a code-switched question (Hindi+English, Tamil+English, etc.).  
The pipeline runs: **VAD → ASR (IndicConformer / Whisper) → RAG → CMI-conditioned TTS**.  
The spoken reply mirrors your language register (Hinglish in → Hinglish out).

In [ ]:
from setu_rag.speech_pipeline import SpeechSetuRAG
voice = SpeechSetuRAG(rag)   # wraps the text pipeline with ASR + TTS
print('Voice pipeline ready')

In [ ]:
import gradio as gr
import numpy as np
from setu_rag.speech.audio_io import from_array

def respond(mic):
    if mic is None:
        return None, 'Speak or upload a clip above.'
    sr, samples = mic
    samples = np.asarray(samples, dtype='float32')
    if samples.ndim == 2:            # stereo → mono
        samples = samples.mean(axis=1)
    if samples.size and np.abs(samples).max() > 1.0:   # int16 → [-1, 1]
        samples = samples / 32768.0
    turn = voice.answer_audio(from_array(samples, sr))
    out = None
    if turn.answer_audio is not None:
        out = (turn.answer_audio.sr, turn.answer_audio.samples)
    label = f'[{turn.matrix_lang}  CMI={turn.cmi:.2f}]  {turn.transcript or "(no transcript)"}'
    return out, f'{label}\n\n{turn.answer_text}'

with gr.Blocks(title='SETU-RAG') as demo:
    gr.Markdown('## SETU-RAG — speak in any Indian language + English')
    gr.Markdown('Try: *"mera refund kab tak aayega"* · *"order track kaise karu"* · *"coupon apply nahi ho raha"*')
    mic     = gr.Audio(sources=['microphone', 'upload'], type='numpy', label='Your question')
    out_aud = gr.Audio(label='Spoken reply', autoplay=True)
    out_txt = gr.Textbox(label='Transcript → answer', lines=4)
    mic.change(respond, inputs=mic, outputs=[out_aud, out_txt])

demo.launch()

## 4 · CS-RAGAS evaluation table
Runs the 5 code-switched sample QA pairs through the RAG pipeline and prints:  
route · measured CMI · retrieval hit@k · CMI-alignment · language-consistency · answer WER/CER · faithfulness.

In [ ]:
from setu_rag.eval.run_eval import main as run_eval
_ = run_eval(eval_path='data/eval.sample.jsonl', pipeline=rag)

## 5 · Optional upgrades (real-with-fallback — install to enhance, skip to degrade gracefully)

| Package | What it enables |
|---|---|
| `ai4bharat-transliteration` | IndicXlit native-script query view |
| `git+https://github.com/AI4Bharat/IndicLID.git` | Token LID covering all 22 langs + romanized |
| `IndicTransToolkit` | IndicTrans2 English-pivot + matrix-canonical query views |
| IndicConformer (NeMo, per its GitHub) | Best code-switched ASR (Whisper auto-fallback) |

In [ ]:
# Uncomment what you want and re-run cells 2–4 after installing:
# !pip -q install ai4bharat-transliteration
# !pip -q install git+https://github.com/AI4Bharat/IndicLID.git
# !pip -q install IndicTransToolkit

# Then enable the English-pivot + matrix-canonical retrieval views:
# rag = build_pipeline(enable_translation=True)
# voice = SpeechSetuRAG(rag)

# Dissertation generator (Sarvam-1 — needs HF token + accepted license):
# from setu_rag.config import SETTINGS
# SETTINGS.prefer_demo_generator = False
# rag = build_pipeline()